# Módulos propios

Agrupar funciones relacionadas en módulos (ficheros `.py`) e importarlos. Cada celda escribe el módulo en un fichero junto al notebook y lo importa.

## 1. Preparación

Función auxiliar para escribir un módulo en disco e importarlo (recargándolo si ya existía).

In [1]:
import importlib
import sys
import textwrap

sys.path.insert(0, ".")

def crea_modulo(nombre, codigo):
    with open(f"{nombre}.py", "w", encoding="utf-8") as f:
        f.write(textwrap.dedent(codigo))
    if nombre in sys.modules:
        return importlib.reload(sys.modules[nombre])
    return importlib.import_module(nombre)

## 2. Módulo `dni`

Módulo con `letra_dni` y `comprueba_letra_dni`, usado desde un programa que pide número y letra y responde `Bienvenido` o `Ha cometido ud. un error`.

In [2]:
dni = crea_modulo("dni", '''
    """Funciones para la letra del DNI."""
    LETRAS = "TRWAGMYFPDXBNJZSQVHLCKE"

    def letra_dni(numero):
        return LETRAS[numero % 23]

    def comprueba_letra_dni(numero, letra):
        return letra_dni(numero) == letra.upper()
''')

numero = int(input("Número de DNI: "))
letra = input("Letra: ")
print("Bienvenido" if dni.comprueba_letra_dni(numero, letra) else "Ha cometido ud. un error")

Número de DNI: 12345678
Letra: Z
Bienvenido


## 3. Módulo `hipotecas`

Módulo documentado con las funciones de hipotecas: cuota mensual, total pagado, intereses y porcentaje de intereses.

In [3]:
hipotecas = crea_modulo("hipotecas", '''
    """Cálculos sobre hipotecas de h euros a n años con un interés anual del i %."""

    def cuota(h, n, i):
        """Cuota mensual redondeada a dos decimales."""
        r = i / (100 * 12)
        return round(h * r / (1 - (1 + r) ** (-12 * n)), 2)

    def total_pagado(h, n, i):
        """Cantidad total pagada al banco."""
        return cuota(h, n, i) * 12 * n

    def intereses(h, n, i):
        """Intereses pagados en euros."""
        return total_pagado(h, n, i) - h

    def porcentaje_intereses(h, n, i):
        """Intereses como porcentaje del capital."""
        return intereses(h, n, i) / h * 100
''')
help(hipotecas.cuota)
print(hipotecas.cuota(150000, 15, 4.75), round(hipotecas.porcentaje_intereses(150000, 15, 4.75), 2))

Help on function cuota in module hipotecas:

cuota(h, n, i)
    Cuota mensual redondeada a dos decimales.

1166.75 40.01


## 4. Módulo `vectores`

Módulo para vectores de dimensión arbitraria: leer un vector, mostrarlo como `(v1, v2, ..., vn)`, longitud y suma (`None` si las dimensiones no coinciden).

In [4]:
vectores = crea_modulo("vectores", '''
    """Operaciones con vectores n-dimensionales representados como listas."""
    from math import sqrt

    def v_lee_vector():
        n = int(input("Dimensión: "))
        return [float(input(f"Componente {i + 1}: ")) for i in range(n)]

    def v_muestra_vector(v):
        print("(" + ", ".join(str(x) for x in v) + ")")

    def v_longitud(v):
        return sqrt(sum(x * x for x in v))

    def v_suma(a, b):
        return [x + y for x, y in zip(a, b)] if len(a) == len(b) else None
''')
v = vectores.v_lee_vector()
vectores.v_muestra_vector(v)
print(vectores.v_longitud(v), vectores.v_suma(v, [1, 1, 1]), vectores.v_suma(v, [1]))

Dimensión: 3
Componente 1: 1
Componente 2: 2
Componente 3: 2
(1.0, 2.0, 2.0)
3.0 [2.0, 3.0, 3.0] None


## 5. Módulo `conjuntos`

Módulo de conjuntos representados como listas sin repetidos: `lista_a_conjunto`, `union`, `interseccion`, `diferencia` e `iguales` (sin importar el orden).

In [5]:
conjuntos = crea_modulo("conjuntos", '''
    """Conjuntos representados como listas sin elementos repetidos."""

    def lista_a_conjunto(lista):
        resultado = []
        for x in lista:
            if x not in resultado:
                resultado.append(x)
        return resultado

    def union(a, b):
        return lista_a_conjunto(a + b)

    def interseccion(a, b):
        return [x for x in a if x in b]

    def diferencia(a, b):
        return [x for x in a if x not in b]

    def iguales(a, b):
        return len(a) == len(b) and all(x in b for x in a)
''')
print(conjuntos.lista_a_conjunto([1, 1, 3, 2, 3]), conjuntos.union([1, 2], [2, 3]), conjuntos.interseccion([1, 2], [2, 3]))
print(conjuntos.diferencia([1, 2], [2, 3]), conjuntos.iguales([1, 3, 2], [2, 1, 3]))

[1, 3, 2] [1, 2, 3] [2]
[1] True


## 6. Módulo `polinomios` con menú

Módulo de polinomios (lista de coeficientes de menor a mayor grado) con suma, resta, producto y representación. ¿Es correcta una `suma` que suma hasta la longitud mínima y añade los restos `a[m:] + b[m:]`? Sí: uno de los dos restos siempre está vacío. Programa con menú: leer y mostrar `a` y `b`, sumar, restar en ambos sentidos y multiplicar.

In [6]:
polinomios = crea_modulo("polinomios", '''
    """Polinomios como listas de coeficientes: [c0, c1, c2] = c0 + c1·x + c2·x²."""

    def normaliza(p):
        p = p[:]
        while len(p) > 1 and p[-1] == 0:
            p.pop()
        return p

    def suma(a, b):
        m = min(len(a), len(b))
        return normaliza([a[i] + b[i] for i in range(m)] + a[m:] + b[m:])

    def resta(a, b):
        return suma(a, [-x for x in b])

    def producto(a, b):
        c = [0] * (len(a) + len(b) - 1)
        for i, x in enumerate(a):
            for j, y in enumerate(b):
                c[i + j] += x * y
        return normaliza(c)

    def texto(p):
        terminos = [f"{c}" + ("" if i == 0 else "x" if i == 1 else f"x^{i}") for i, c in enumerate(p) if c]
        return " + ".join(reversed(terminos)) or "0"
''')

a = b = [0]
while (opcion := input("Opción (1-9): ")) != "9":
    if opcion in ("1", "3"):
        p = [float(c) for c in input("Coeficientes (de menor a mayor grado): ").split()]
        a, b = (p, b) if opcion == "1" else (a, p)
    elif opcion in ("2", "4"):
        print(polinomios.texto(a if opcion == "2" else b))
    elif opcion == "5":
        print(polinomios.texto(polinomios.suma(a, b)))
    elif opcion == "6":
        print(polinomios.texto(polinomios.resta(b, a)))
    elif opcion == "7":
        print(polinomios.texto(polinomios.resta(a, b)))
    elif opcion == "8":
        print(polinomios.texto(polinomios.producto(a, b)))

Opción (1-9): 1
Coeficientes (de menor a mayor grado): 1 2
Opción (1-9): 3
Coeficientes (de menor a mayor grado): 0 1 1
Opción (1-9): 2
2.0x + 1.0
Opción (1-9): 5
1.0x^2 + 3.0x + 1.0
Opción (1-9): 6
1.0x^2 + -1.0x + -1.0
Opción (1-9): 7
-1.0x^2 + 1.0x + 1.0
Opción (1-9): 8
2.0x^3 + 3.0x^2 + 1.0x
Opción (1-9): 9


## 7. Módulo `estadisticas`

Módulo de estadísticas que trate correctamente las listas vacías, con media, desviación típica, coeficiente de variación ($\sigma / \bar{x}$) y recorrido (máximo − mínimo).

In [7]:
estadisticas = crea_modulo("estadisticas", '''
    """Estadísticos de una lista de números. Con la lista vacía devuelven 0."""

    def media(lista):
        return sum(lista) / len(lista) if lista else 0

    def desviacion(lista):
        if not lista:
            return 0
        m = media(lista)
        return (sum((x - m) ** 2 for x in lista) / len(lista)) ** 0.5

    def coeficiente_variacion(lista):
        m = media(lista)
        return desviacion(lista) / m if m else 0

    def recorrido(lista):
        return max(lista) - min(lista) if lista else 0
''')
datos = [2, 4, 4, 4, 5, 5, 7, 9]
print(estadisticas.media(datos), estadisticas.coeficiente_variacion(datos), estadisticas.recorrido(datos), estadisticas.media([]))

5.0 0.4 7 0


## 8. Módulo `matrices`

Módulo de matrices con `dimension`, `es_cuadrada` (usando `dimension`) y `producto` (`None` si no se pueden multiplicar).

In [8]:
matrices = crea_modulo("matrices", '''
    """Matrices como listas de filas."""

    def dimension(m):
        return len(m), len(m[0]) if m else 0

    def es_cuadrada(m):
        filas, columnas = dimension(m)
        return filas == columnas

    def producto(a, b):
        if dimension(a)[1] != dimension(b)[0]:
            return None
        return [[sum(x * y for x, y in zip(fila, col)) for col in zip(*b)] for fila in a]
''')
A, B = [[1, 2, 3], [4, 5, 6]], [[-1, 0], [0, 1], [1, 1]]
print(matrices.es_cuadrada(A), matrices.es_cuadrada([[1, 2], [3, 4]]), matrices.producto(A, B), matrices.producto(A, A))

False True [[2, 5], [2, 11]] None
